## Projet - Prediction des ventes hebdomadaire

## Dataset Walmart sales forecast

### Objectif

Le projet vise à consevoir un système intelligent capable de prévoir les ventes hebdomadaires dans les magasins de distribution à partir des données historiques. Le système exploitera les données historiques des ventes, les patterns temporels et saisonniers.

Plan du notebook:

1. Préparation de l'environnement de travail
2. Chargement, fusion et inspection du dataset
3. Analyse Exploratoire
4. Feature Engineering
5. Modélisation et évaluation
7. Deploiement



### 1. Preparation de l'environnement de travail

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit,learning_curve
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
        Dense, Dropout, LSTM, Conv1D, MaxPooling1D, Flatten, BatchNormalization
    )
pd.set_option("display.max_columns",None)


### 2. Chargement, Inspection et Fusion des tables et fusion

### 2.1.  Chargement et inspection

In [2]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
store = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")

In [3]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Store         421570 non-null  int64  
 1   Dept          421570 non-null  int64  
 2   Date          421570 non-null  str    
 3   Weekly_Sales  421570 non-null  float64
 4   IsHoliday     421570 non-null  bool   
dtypes: bool(1), float64(1), int64(2), str(1)
memory usage: 13.3 MB


In [4]:
train.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


In [5]:
#Affichons les differentes boutiques
print(f"Listons les boutiques enregistrées dans le fichier train")
train["Store"].unique()

Listons les boutiques enregistrées dans le fichier train


array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45])

In [6]:
#Affichons les informations de la table store
print("Affichons les informations sommaires de la table store")
store.info()

Affichons les informations sommaires de la table store
<class 'pandas.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Store   45 non-null     int64
 1   Type    45 non-null     str  
 2   Size    45 non-null     int64
dtypes: int64(2), str(1)
memory usage: 1.2 KB


In [7]:
#Affichons les 5 premieres lignes
store.head()

,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


In [8]:
print("Affichons toutes les modalités disponible de la variable Store")
store["Store"].unique()

Affichons toutes les modalités disponible de la variable Store


array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45])

In [9]:
features.info()

<class 'pandas.DataFrame'>
RangeIndex: 8190 entries, 0 to 8189
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         8190 non-null   int64  
 1   Date          8190 non-null   str    
 2   Temperature   8190 non-null   float64
 3   Fuel_Price    8190 non-null   float64
 4   MarkDown1     4032 non-null   float64
 5   MarkDown2     2921 non-null   float64
 6   MarkDown3     3613 non-null   float64
 7   MarkDown4     3464 non-null   float64
 8   MarkDown5     4050 non-null   float64
 9   CPI           7605 non-null   float64
 10  Unemployment  7605 non-null   float64
 11  IsHoliday     8190 non-null   bool   
dtypes: bool(1), float64(9), int64(1), str(1)
memory usage: 712.0 KB


In [10]:
features.head()

,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,2010-02-26,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,2010-03-05,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


####  2.2.  Fusion des dataset

### A. Fusion du fichiers train, features et stores

In [11]:
df_train=pd.merge(train,
              store,
              how="left",
              on="Store",
                 indicator=True)
assert (df_train["_merge"]=="both").all()  #Cela nous permettra de verifier si tous les stores repertoriés dans train ont une correspondance dans Store
df_train=df_train.drop("_merge",axis=1) #Suppression de la colonne _merge après vérification
df_train.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size
0,1,1,2010-02-05,24924.50,False,A,151315
1,1,1,2010-02-12,46039.49,True,A,151315
2,1,1,2010-02-19,41595.55,False,A,151315
3,1,1,2010-02-26,19403.54,False,A,151315
4,1,1,2010-03-05,21827.90,False,A,151315


In [12]:
df_train=pd.merge(df_train,
                 features,
                 how="left",
                 on=["Store","Date","IsHoliday"],
                 indicator=True)
#Statisques de fusion
merge_stats=df_train["_merge"].value_counts()
print("Statistiques de fusion ")
print(merge_stats)
print(f"Ligne de la table Train ayant héritées des NaN après fusion {(df_train["_merge"]=="Left_only").sum()} ")
df_train=df_train.drop("_merge",axis=1)   #Suppression de la colonne d'audit
#Vérifions si les Holiday_x et Holiday_y correspondent
if "IsHoliday_x" in df_train.columns and "IsHoliday_y" in df_train.columns:
    no_matching=(df_train["IsHoliday_x"]!=df_train["IsHoliday_y"])
    if no_matching.sum()>0:
        print(f"Attention: { no_matching.sum() } lignes de IsHoliday_x ne correspondent pas à IsHoliday_y")
    # Harmonisation
    df_train["IsHoliday"]=df_train["IsHoliday_x"]| df_train["IsHoliday_y"]
    df_train=df_train.drop(["IsHoliday_x","IsHoliday_y"],axis=1)


df_train.info()

Statistiques de fusion 
_merge
both          421570
left_only          0
right_only         0
Name: count, dtype: int64
Ligne de la table Train ayant héritées des NaN après fusion 0 
<class 'pandas.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Store         421570 non-null  int64  
 1   Dept          421570 non-null  int64  
 2   Date          421570 non-null  str    
 3   Weekly_Sales  421570 non-null  float64
 4   IsHoliday     421570 non-null  bool   
 5   Type          421570 non-null  str    
 6   Size          421570 non-null  int64  
 7   Temperature   421570 non-null  float64
 8   Fuel_Price    421570 non-null  float64
 9   MarkDown1     150681 non-null  float64
 10  MarkDown2     111248 non-null  float64
 11  MarkDown3     137091 non-null  float64
 12  MarkDown4     134967 non-null  float64
 13  MarkDown5     151432 non-null  float64
 14  CPI     

B. Fusion du fichier test features et stores

In [13]:
test.head()

,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


In [14]:
df_test=pd.merge(test,
              store,
              how="left",
              on="Store",
                 indicator=True)
assert (df_test["_merge"]=="both").all()  #Cela nous permettra de verifier si tous les stores repertoriés dans train ont une correspondance dans Store
df_test=df_test.drop("_merge",axis=1) #Suppression de la colonne _merge après vérification
df_test.head()

,Store,Dept,Date,IsHoliday,Type,Size
0,1,1,2012-11-02,False,A,151315
1,1,1,2012-11-09,False,A,151315
2,1,1,2012-11-16,False,A,151315
3,1,1,2012-11-23,True,A,151315
4,1,1,2012-11-30,False,A,151315


In [15]:
df_test=pd.merge(df_test,
                 features,
                 how="left",
                 on=["Store","Date"],
                 indicator=True)
#Statisques de fusion
merge_stats=df_test["_merge"].value_counts()
print("Statistiques de fusion ")
print(merge_stats)
print(f"Ligne de la table Test ayant héritées des NaN après fusion {(df_test["_merge"]=="Left_only").sum()} ")
df_test=df_test.drop("_merge",axis=1)   #Suppression de la colonne d'audit
#Vérifions si les Holiday_x et Holiday_y correspondent
if "IsHoliday_x" in df_test.columns and "IsHoliday_y" in df_test.columns:
    no_matching=(df_test["IsHoliday_x"]!=df_test["IsHoliday_y"])
    if no_matching.sum()>0:
        print(f"Attention: { no_matching.sum() } lignes de IsHoliday_x ne correspondent pas à IsHoliday_y")
    # Harmonisation
    df_test["IsHoliday"]=df_test["IsHoliday_x"]| df_test["IsHoliday_y"]
    df_test=df_test.drop(["IsHoliday_x","IsHoliday_y"],axis=1)


df_test.head()

Statistiques de fusion 
_merge
both          115064
left_only          0
right_only         0
Name: count, dtype: int64
Ligne de la table Test ayant héritées des NaN après fusion 0 


,Store,Dept,Date,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,1,2012-11-02,A,151315,55.32,3.386,6766.44,5147.70,50.82,3639.90,2737.42,223.462779,6.573,False
1,1,1,2012-11-09,A,151315,61.24,3.314,11421.32,3370.89,40.28,4646.79,6154.16,223.481307,6.573,False
2,1,1,2012-11-16,A,151315,52.92,3.252,9696.28,292.10,103.78,1133.15,6612.69,223.512911,6.573,False
3,1,1,2012-11-23,A,151315,56.23,3.211,883.59,4.17,74910.32,209.91,303.32,223.561947,6.573,True
4,1,1,2012-11-30,A,151315,52.34,3.207,2460.03,NaN,3838.35,150.57,6966.34,223.610984,6.573,False


In [16]:
df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 115064 entries, 0 to 115063
Data columns (total 15 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Store         115064 non-null  int64  
 1   Dept          115064 non-null  int64  
 2   Date          115064 non-null  str    
 3   Type          115064 non-null  str    
 4   Size          115064 non-null  int64  
 5   Temperature   115064 non-null  float64
 6   Fuel_Price    115064 non-null  float64
 7   MarkDown1     114915 non-null  float64
 8   MarkDown2     86437 non-null   float64
 9   MarkDown3     105235 non-null  float64
 10  MarkDown4     102176 non-null  float64
 11  MarkDown5     115064 non-null  float64
 12  CPI           76902 non-null   float64
 13  Unemployment  76902 non-null   float64
 14  IsHoliday     115064 non-null  bool   
dtypes: bool(1), float64(9), int64(3), str(2)
memory usage: 12.4 MB
